In [1]:
!nvidia-smi
!pip install nvidia-pyindex
!pip install tensorrt==10.0.1
!pip install onnx onnxruntime
!pip install pycuda
!pip install optimum[onnxruntime-gpu]
!pip install transformers numpy

Wed Dec 24 14:10:45 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
"""
BERT-base (google-bert/bert-base-uncased)
PyTorch FP16 vs TensorRT FP16 inference benchmark
Masked Language Modeling (encoder-only)
"""

import os
import time
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForMaskedLM

import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit


# -----------------------
# Config
# -----------------------
MODEL_ID = "google-bert/bert-base-uncased"

ONNX_PATH = "bert_base.onnx"
ENGINE_PATH = "bert_base_fp16.engine"

BATCH = 1
SEQ_LEN = 32

MIN_SHAPE = (1, 8)
OPT_SHAPE = (1, 32)
MAX_SHAPE = (1, 128)

WARMUP = 20
ITERS = 100
TRT_WORKSPACE_GB = 4


# -----------------------
# Utils
# -----------------------
def pad_input(tokenizer, input_ids, seq_len, device=None):
    if device:
        input_ids = input_ids.to(device)

    if input_ids.shape[1] < seq_len:
        pad = torch.full(
            (input_ids.shape[0], seq_len - input_ids.shape[1]),
            tokenizer.pad_token_id,
            dtype=input_ids.dtype,
            device=input_ids.device,
        )
        input_ids = torch.cat([input_ids, pad], dim=1)
    else:
        input_ids = input_ids[:, :seq_len]

    return input_ids


def torch_latency(model, input_ids):
    model.eval()
    with torch.inference_mode():
        for _ in range(WARMUP):
            _ = model(input_ids=input_ids)
        torch.cuda.synchronize()

        t0 = time.time()
        for _ in range(ITERS):
            _ = model(input_ids=input_ids)
        torch.cuda.synchronize()

    return (time.time() - t0) / ITERS


# -----------------------
# ONNX export (CPU, STABLE)
# -----------------------
def export_onnx_cpu(tokenizer, seq_len):
    model = AutoModelForMaskedLM.from_pretrained(MODEL_ID).eval().to("cpu")

    dummy_text = "TensorRT optimization is powerful"
    input_ids = tokenizer(dummy_text, return_tensors="pt").input_ids

    input_ids = pad_input(tokenizer, input_ids, seq_len)

    torch.onnx.export(
        model,
        (input_ids,),
        ONNX_PATH,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq_len"},
            "logits": {0: "batch", 1: "seq_len"},
        },
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )


# -----------------------
# TensorRT build (TRT 9/10)
# -----------------------
def build_trt_engine(onnx_path, engine_path):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    runtime = trt.Runtime(logger)

    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)

    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parse failed")

    config = builder.create_builder_config()
    config.set_flag(trt.BuilderFlag.FP16)
    config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE, TRT_WORKSPACE_GB << 30
    )

    input_name = network.get_input(0).name
    profile = builder.create_optimization_profile()
    profile.set_shape(input_name, MIN_SHAPE, OPT_SHAPE, MAX_SHAPE)
    config.add_optimization_profile(profile)

    serialized_engine = builder.build_serialized_network(network, config)
    if serialized_engine is None:
        raise RuntimeError("TensorRT engine build failed")

    engine = runtime.deserialize_cuda_engine(serialized_engine)

    with open(engine_path, "wb") as f:
        f.write(serialized_engine)

    return engine


# -----------------------
# TensorRT latency
# -----------------------
def trt_latency(engine, input_ids_np):
    context = engine.create_execution_context()

    input_name = engine.get_tensor_name(0)
    output_name = engine.get_tensor_name(1)

    context.set_input_shape(input_name, input_ids_np.shape)
    out_shape = context.get_tensor_shape(output_name)

    d_input = cuda.mem_alloc(input_ids_np.nbytes)

    out_dtype = np.float16 if engine.get_tensor_dtype(output_name) == trt.DataType.HALF else np.float32
    d_output = cuda.mem_alloc(trt.volume(out_shape) * np.dtype(out_dtype).itemsize)

    stream = cuda.Stream()

    context.set_tensor_address(input_name, int(d_input))
    context.set_tensor_address(output_name, int(d_output))

    for _ in range(WARMUP):
        cuda.memcpy_htod_async(d_input, input_ids_np, stream)
        context.execute_async_v3(stream.handle)
    stream.synchronize()

    t0 = time.time()
    for _ in range(ITERS):
        cuda.memcpy_htod_async(d_input, input_ids_np, stream)
        context.execute_async_v3(stream.handle)
    stream.synchronize()

    return (time.time() - t0) / ITERS


# -----------------------
# Main
# -----------------------
def main():
    torch.backends.cudnn.benchmark = True
    device = "cuda"

    print("Loading model:", MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model = AutoModelForMaskedLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
    ).to(device).eval()

    text = "TensorRT optimization is powerful"
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    input_ids = pad_input(tokenizer, input_ids, SEQ_LEN, device=device)

    print("Input shape:", input_ids.shape)

    pt_lat = torch_latency(model, input_ids)
    print(f"PyTorch FP16 latency: {pt_lat * 1000:.3f} ms")

    if not os.path.exists(ONNX_PATH):
        print("Exporting ONNX (CPU)...")
        export_onnx_cpu(tokenizer, SEQ_LEN)

    if not os.path.exists(ENGINE_PATH):
        print("Building TensorRT engine...")
        engine = build_trt_engine(ONNX_PATH, ENGINE_PATH)
    else:
        runtime = trt.Runtime(trt.Logger(trt.Logger.WARNING))
        with open(ENGINE_PATH, "rb") as f:
            engine = runtime.deserialize_cuda_engine(f.read())

    trt_lat = trt_latency(engine, input_ids.cpu().numpy().astype(np.int32))
    print(f"TensorRT FP16 latency: {trt_lat * 1000:.3f} ms")

    print(f"Speedup: {pt_lat / trt_lat:.2f}x")


if __name__ == "__main__":
    main()


Loading model: google-bert/bert-base-uncased


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Input shape: torch.Size([1, 32])
PyTorch FP16 latency: 7.862 ms
Exporting ONNX (CPU)...


Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/tmp/ipython-input-3558182094.py:86: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporti

Building TensorRT engine...
TensorRT FP16 latency: 0.831 ms
Speedup: 9.47x


In [9]:
import os
import time
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForMaskedLM

import onnxruntime as ort
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit


# =========================
# CONFIG
# =========================
MODEL_ID = "google-bert/bert-base-uncased"

ONNX_PATH = "bert_base.onnx"
ENGINE_PATH = "bert_base_fp16.engine"

BATCH = 1
SEQ_LEN = 32

MIN_SHAPE = (1, 8)
OPT_SHAPE = (1, 32)
MAX_SHAPE = (1, 128)

WARMUP = 20
ITERS = 100
TRT_WORKSPACE_GB = 4


# =========================
# UTILS
# =========================
def pad_input(tokenizer, input_ids, seq_len, device=None):
    if device:
        input_ids = input_ids.to(device)

    if input_ids.shape[1] < seq_len:
        pad = torch.full(
            (input_ids.shape[0], seq_len - input_ids.shape[1]),
            tokenizer.pad_token_id,
            dtype=input_ids.dtype,
            device=input_ids.device,
        )
        input_ids = torch.cat([input_ids, pad], dim=1)
    else:
        input_ids = input_ids[:, :seq_len]

    return input_ids


# =========================
# PYTORCH
# =========================
def measure_torch_latency(model, input_ids):
    model.eval()
    with torch.inference_mode():
        for _ in range(WARMUP):
            _ = model(input_ids=input_ids)
        torch.cuda.synchronize()

        t0 = time.time()
        for _ in range(ITERS):
            _ = model(input_ids=input_ids)
        torch.cuda.synchronize()

    latency = (time.time() - t0) / ITERS
    throughput = BATCH / latency
    return latency, throughput


# =========================
# ONNX RUNTIME
# =========================
def export_onnx_cpu(tokenizer):
    print("Exporting ONNX (CPU)...")
    model = AutoModelForMaskedLM.from_pretrained(MODEL_ID).eval().cpu()

    dummy = tokenizer(
        "TensorRT optimization is powerful",
        return_tensors="pt"
    ).input_ids
    dummy = pad_input(tokenizer, dummy, SEQ_LEN)

    torch.onnx.export(
        model,
        (dummy,),
        ONNX_PATH,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq_len"},
            "logits": {0: "batch", 1: "seq_len"},
        },
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )


def measure_onnxruntime_latency(onnx_path, input_ids_np):
    sess = ort.InferenceSession(
        onnx_path,
        providers=["CUDAExecutionProvider"]
    )

    input_name = sess.get_inputs()[0].name

    for _ in range(WARMUP):
        sess.run(None, {input_name: input_ids_np})

    t0 = time.time()
    for _ in range(ITERS):
        sess.run(None, {input_name: input_ids_np})

    latency = (time.time() - t0) / ITERS
    throughput = BATCH / latency
    return latency, throughput


# =========================
# TENSORRT
# =========================
def build_trt_engine(onnx_path):
    print("Building TensorRT engine (FP16)...")

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    runtime = trt.Runtime(logger)

    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)

    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parse failed")

    config = builder.create_builder_config()
    config.set_flag(trt.BuilderFlag.FP16)
    config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE,
        TRT_WORKSPACE_GB << 30
    )

    input_name = network.get_input(0).name
    profile = builder.create_optimization_profile()
    profile.set_shape(input_name, MIN_SHAPE, OPT_SHAPE, MAX_SHAPE)
    config.add_optimization_profile(profile)

    serialized_engine = builder.build_serialized_network(network, config)
    if serialized_engine is None:
        raise RuntimeError("TensorRT engine build failed")

    with open(ENGINE_PATH, "wb") as f:
        f.write(serialized_engine)

    return runtime.deserialize_cuda_engine(serialized_engine)


def measure_trt_latency(engine, input_ids_np):
    context = engine.create_execution_context()

    input_name = engine.get_tensor_name(0)
    output_name = engine.get_tensor_name(1)

    context.set_input_shape(input_name, input_ids_np.shape)
    out_shape = context.get_tensor_shape(output_name)

    d_input = cuda.mem_alloc(input_ids_np.nbytes)
    output_size = int(np.prod(out_shape)) * np.dtype(np.float16).itemsize
    d_output = cuda.mem_alloc(output_size)

    stream = cuda.Stream()

    context.set_tensor_address(input_name, int(d_input))
    context.set_tensor_address(output_name, int(d_output))

    for _ in range(WARMUP):
        cuda.memcpy_htod_async(d_input, input_ids_np, stream)
        context.execute_async_v3(stream.handle)
    stream.synchronize()

    t0 = time.time()
    for _ in range(ITERS):
        cuda.memcpy_htod_async(d_input, input_ids_np, stream)
        context.execute_async_v3(stream.handle)
    stream.synchronize()

    latency = (time.time() - t0) / ITERS
    throughput = BATCH / latency
    return latency, throughput


# =========================
# MAIN
# =========================
def main():
    torch.backends.cudnn.benchmark = True
    device = "cuda"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model = AutoModelForMaskedLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16
    ).to(device).eval()

    text = "TensorRT optimization is powerful"
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    input_ids = pad_input(tokenizer, input_ids, SEQ_LEN, device=device)

    print("Input shape:", input_ids.shape)

    # PyTorch
    pt_lat, pt_thr = measure_torch_latency(model, input_ids)
    print(f"PyTorch FP16 latency: {pt_lat*1000:.3f} ms | Throughput: {pt_thr:.1f} seq/s")

    # ONNX Runtime
    if not os.path.exists(ONNX_PATH):
        export_onnx_cpu(tokenizer)

    ort_lat, ort_thr = measure_onnxruntime_latency(
        ONNX_PATH,
        input_ids.cpu().numpy().astype(np.int64)
    )
    print(f"ONNX Runtime CUDA latency: {ort_lat*1000:.3f} ms | Throughput: {ort_thr:.1f} seq/s")

    # TensorRT
    if os.path.exists(ENGINE_PATH):
        runtime = trt.Runtime(trt.Logger(trt.Logger.WARNING))
        with open(ENGINE_PATH, "rb") as f:
            engine = runtime.deserialize_cuda_engine(f.read())
    else:
        engine = build_trt_engine(ONNX_PATH)

    trt_lat, trt_thr = measure_trt_latency(
        engine,
        input_ids.cpu().numpy().astype(np.int32)
    )
    print(f"TensorRT FP16 latency: {trt_lat*1000:.3f} ms | Throughput: {trt_thr:.1f} seq/s")

    print("\n===== SPEEDUPS =====")
    print(f"ONNX vs PyTorch: {pt_lat / ort_lat:.2f}x")
    print(f"TensorRT vs PyTorch: {pt_lat / trt_lat:.2f}x")


if __name__ == "__main__":
    main()


Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Input shape: torch.Size([1, 32])
PyTorch FP16 latency: 8.156 ms | Throughput: 122.6 seq/s
ONNX Runtime CUDA latency: 4.123 ms | Throughput: 242.5 seq/s
TensorRT FP16 latency: 0.700 ms | Throughput: 1428.7 seq/s

===== SPEEDUPS =====
ONNX vs PyTorch: 1.98x
TensorRT vs PyTorch: 11.65x
